### Zimmerbelegung – Lösung Schritt für Schritt

*36. Bundeswettbewerb Informatik (2017), 1. Runde, Aufgabe 1*

Die Klasse 9c des Alan-Turing-Gymnasiums, einer Mädchenschule, möchte eine Klassenfahrt unternehmen. Die wichtigste Frage dabei: Wer teilt sich mit wem ein Zimmer? Das ist leider gar nicht so einfach zu klären, denn die Wünsche der Schülerinnen können sich widersprechen.

Bei der letzten Klassenfahrt wollte zum Beispiel Anna gerne mit Paula ein Zimmer teilen, aber Paula nicht mit Anna. Und dann wollten sowohl Dani als auch Lotta gerne mit Steffi ein Zimmer teilen, aber auf keinen Fall miteinander. Da gab es natürlich keine Möglichkeit, alle Wünsche zu erfüllen.

Die Klassenlehrerin der 9c möchte herausfinden, ob dieses Mal alle Wünsche erfüllt werden können und wie in diesem Fall die Zimmerbelegung aussieht. Hilf der Lehrerin!

#### Aufgabe 1

Schreibe ein Programm, das ermittelt, ob alle Wünsche erfüllt werden können, wenn es genug Zimmer jeder Größe gibt. Als Eingabe erhält es für jede Schülerin zwei Listen der Mitschülerinnen, mit denen sie auf jeden Fall (+) bzw. auf keinen Fall (–) ein Zimmer teilen möchte.

Dein Programm soll ausgeben, ob eine Zimmerbelegung möglich ist, die alle Wünsche erfüllt. Falls ja, soll es zusätzlich eine solche Zimmerbelegung ausgeben. Zum Einstieg könntest du zunächst annehmen, dass jede Schülerin jede Mitschülerin in einer ihrer beiden Listen nennt.

Wende dein Programm auf die Beispiele an, die du auf den BwInf-Webseiten findest.

#### Eingabeformat (`beispieldaten/zimmerbelegung1.txt` bis `zimmerbelegung6.txt`)

Je Schülerin ein Block aus drei Zeilen und einer Leerzeile: der Name, eine Zeile `+ ` mit den Mitschülerinnen, mit denen sie ein Zimmer teilen will, und eine Zeile `- ` mit denen, mit denen sie auf keinen Fall eines teilen will (Namen durch Leerzeichen getrennt, die Listen können leer sein). Wen sie in keiner Liste nennt, dem gegenüber hat sie keinen Wunsch („ist ihr egal“).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zwingende Folgerungen:** Will Anna mit Paula ein Zimmer teilen und Paula mit Steffi, dann sind Anna, Paula und Steffi **zwingend** im selben Zimmer, auch wenn Anna und Steffi nie über einander gesprochen haben. Man sammelt alles, was sich auf diese Weise ergibt.
- **Flutfüllen (Breitensuche im Graphen):** Von einer Schülerin aus alle erreichen, die über „will zusammen“-Wünsche verbunden sind. Dazu braucht man eine Warteschlange und die Menge der schon besuchten.
- **Eine Aufgabe in zwei Schritte zerlegen:** erst die Zimmer bilden, die sich zwingend ergeben, dann prüfen, ob in keinem Zimmer ein „auf keinen Fall“ verletzt wird.
- **Kontrolle mit einem unabhängigen Test:** Eine gefundene Belegung kann man einfach gegen die ursprünglichen Listen prüfen. Und bei kleinen Klassen kann man zur Kontrolle wirklich alle Belegungen durchprobieren.

**Nützliche Funktionen und Bibliotheken**
- Mengen (`set`) für die Wunschlisten: Vereinigung `|`, schnelles Nachschlagen mit `in`, `add` und `discard`.
- Wörterbücher (`dict`) für „Name → Menge von Namen“.
- `random` für Zufallstests am Ende.

## Die Idee

Zwei Arten von Wünschen verhalten sich sehr verschieden:

- **„Auf jeden Fall zusammen“** zwingt Schülerinnen **in dasselbe** Zimmer, und das überträgt sich: Wer mit einer Schülerin zusammen sein muss, muss auch mit allen zusammen sein, mit denen diese zusammen sein muss.
- **„Auf keinen Fall zusammen“** ist nur eine **Prüfung**: Sie verbietet ein Paar, erzwingt aber nichts.

Daraus folgt ein Verfahren in drei Schritten:

1. **Wünsche in beide Richtungen lesen.** Will Anna mit Paula zusammen sein, dann müssen auch Paula und Anna zusammen sein, egal was Paula selbst wünscht (steht dort „nicht“, ist es ein Widerspruch). Genauso, wenn Anna nicht mit Paula zusammen sein will: Dann dürfen auch Paula und Anna nicht zusammen sein. Ist Paula die Sache egal, hat sie ja nichts dagegen.
2. **Die zwingenden Zimmer bilden:** Mit „will zusammen“ als Verbindung bilden alle, die über solche Verbindungen erreichbar sind, ein Zimmer. Kleiner kann man die Zimmer nicht machen, ohne einen Wunsch zu verletzen.
3. **Prüfen:** Steht in einem dieser Zimmer ein Paar, das nicht zusammen sein will, gibt es **keine** Belegung, die alle Wünsche erfüllt: Die beiden müssen nach Schritt 2 zwingend zusammen sein. Sonst ist die Belegung gültig.

Wer nach der Belegung mit den **wenigsten** Zimmern fragt (steht nicht in der Aufgabe), kann am Ende noch Zimmer zusammenlegen; das machen wir bei „Für Neugierige“.

## Schritt 1: Daten einlesen

Wir lesen jeden Dreierblock und merken uns für jede Schülerin zwei Mengen: `will[n]` (mit wem sie zusammen sein will) und `nicht[n]` (mit wem auf keinen Fall). Ein Blick in die Dateien zeigt: In einigen Beispielen nennt eine Schülerin **sich selbst** in einer Liste (z. B. in `zimmerbelegung3.txt` Nele und Larissa). Das ist sinnlos, deshalb lassen wir solche Einträge weg und zählen sie nur.

In [ ]:
def lies_wuensche(dateiname):
    """Gibt (namen, will, nicht, selbst) zurück: die Namen in der Reihenfolge der Datei, je Name zwei Mengen von Namen
    und die Zahl der weggelassenen Selbstnennungen."""
    namen = []
    will = {}
    nicht = {}
    selbst = 0
    with open(dateiname, encoding='utf-8') as datei:
        text = datei.read()
    for block in text.strip().split('\n\n'):
        zeilen = block.split('\n')
        name = zeilen[0].strip()
        namen.append(name)
        will[name] = set(zeilen[1].split()[1:])
        nicht[name] = set(zeilen[2].split()[1:])
        selbst = selbst + (name in will[name]) + (name in nicht[name])
        will[name].discard(name)                  # Selbstnennungen weglassen
        nicht[name].discard(name)
    return namen, will, nicht, selbst


for nummer in range(1, 7):
    namen, will, nicht, selbst = lies_wuensche('beispieldaten/zimmerbelegung' + str(nummer) + '.txt')
    anzahl_will = sum(len(v) for v in will.values())
    anzahl_nicht = sum(len(v) for v in nicht.values())
    print('zimmerbelegung' + str(nummer) + ':', len(namen), 'Schülerinnen,', anzahl_will, 'mal "+",', anzahl_nicht,
          'mal "-", Selbstnennungen weggelassen:', selbst)

zimmerbelegung1: 7 Schülerinnen, 5 mal "+", 3 mal "-", Selbstnennungen weggelassen: 0
zimmerbelegung2: 6 Schülerinnen, 4 mal "+", 4 mal "-", Selbstnennungen weggelassen: 0
zimmerbelegung3: 43 Schülerinnen, 41 mal "+", 43 mal "-", Selbstnennungen weggelassen: 2
zimmerbelegung4: 43 Schülerinnen, 76 mal "+", 62 mal "-", Selbstnennungen weggelassen: 2
zimmerbelegung5: 43 Schülerinnen, 8 mal "+", 12 mal "-", Selbstnennungen weggelassen: 1
zimmerbelegung6: 43 Schülerinnen, 76 mal "+", 74 mal "-", Selbstnennungen weggelassen: 4


## Schritt 2: Wünsche in beide Richtungen

Aus den Listen jeder Schülerin bauen wir zwei Wörterbücher, in denen **jeder Wunsch für beide Beteiligten** eingetragen ist. `zusammen[n]` sind alle, die mit `n` zusammen sein müssen (weil `n` es will oder sie es wollen), `getrennt[n]` alle, die nicht mit `n` zusammen sein dürfen.

Ein Beispiel: In `zimmerbelegung1.txt` will Anna mit Paula zusammen, Paula nicht mit Anna. Danach stehen Paula und Anna in `zusammen[...]` **und** in `getrennt[...]`. Das ist der Widerspruch aus der Aufgabe.

In [ ]:
def beidseitig(namen, will, nicht):
    """Gibt zwei Wörterbücher zurück: zusammen[n] und getrennt[n] enthalten jeweils beide Richtungen des Wunsches."""
    zusammen = {}
    getrennt = {}
    for n in namen:
        zusammen[n] = set()
        getrennt[n] = set()
    for n in namen:
        for m in will[n]:
            zusammen[n].add(m)
            zusammen[m].add(n)
        for m in nicht[n]:
            getrennt[n].add(m)
            getrennt[m].add(n)
    return zusammen, getrennt


namen1, will1, nicht1, _ = lies_wuensche('beispieldaten/zimmerbelegung1.txt')
zusammen1, getrennt1 = beidseitig(namen1, will1, nicht1)
for n in namen1:
    print(n.ljust(7), 'zusammen:', sorted(zusammen1[n]), ' getrennt:', sorted(getrennt1[n]))

Anna    zusammen: ['Paula']  getrennt: ['Paula']
Paula   zusammen: ['Anna']  getrennt: ['Anna']
Dani    zusammen: ['Steffi']  getrennt: ['Lotta']
Lotta   zusammen: ['Steffi']  getrennt: ['Dani']
Steffi  zusammen: ['Cleo', 'Dani', 'Jo', 'Lotta']  getrennt: []
Cleo    zusammen: ['Steffi']  getrennt: []
Jo      zusammen: ['Steffi']  getrennt: []


## Schritt 3: Zimmer bilden (Flutfüllen)

Wir nehmen eine Schülerin, die noch kein Zimmer hat, und sammeln alle, die mit ihr zusammen sein müssen, dann alle, die mit diesen zusammen sein müssen, und so weiter. Dazu dient eine Liste `warteschlange` der noch nicht „abgearbeiteten“ Schülerinnen:

1. Nimm die erste aus der Warteschlange und setze sie ins Zimmer.
2. Hänge alle ihre `zusammen`-Partnerinnen hinten an, die weder im Zimmer sind noch schon warten.
3. Weiter, bis die Warteschlange leer ist.

Dann nehmen wir die nächste Schülerin ohne Zimmer und beginnen von vorn.

In [ ]:
def fuelle_zimmer(start, zusammen):
    """Gibt alle zurück, die (direkt oder über andere) zwingend mit start im selben Zimmer sind."""
    zimmer = []
    warteschlange = [start]
    while len(warteschlange) > 0:
        n = warteschlange.pop(0)            # die erste nehmen
        zimmer.append(n)
        for m in sorted(zusammen[n]):
            if m not in zimmer and m not in warteschlange:
                warteschlange.append(m)
    return zimmer


def bilde_zimmer(namen, zusammen):
    """Gibt die Zimmer als Liste von Listen zurück; jede Schülerin steht in genau einem."""
    zimmer_liste = []
    vergeben = set()
    for n in namen:
        if n not in vergeben:
            zimmer = fuelle_zimmer(n, zusammen)
            zimmer_liste.append(zimmer)
            vergeben = vergeben | set(zimmer)
    return zimmer_liste


for zimmer in bilde_zimmer(namen1, zusammen1):
    print(zimmer)

['Anna', 'Paula']
['Dani', 'Steffi', 'Cleo', 'Jo', 'Lotta']


Bei Beispiel 1 ergeben sich zwei „Pflichtzimmer“: eines mit Anna und Paula, ein zweites mit den übrigen fünf. Schon hier sieht man die Widersprüche, die der nächste Schritt meldet.

## Schritt 4: Prüfen

Jetzt suchen wir in jedem Zimmer ein Paar, das nicht zusammen sein darf. Gibt es keines, sind alle Wünsche erfüllt. Gibt es eines, dann **kann** es keine Belegung geben, denn die beiden sind zwingend zusammen. Damit die Lehrerin sieht, **warum**, geben wir die Kette der „will zusammen“-Verbindungen von der einen zur anderen aus. Die Kette suchen wir mit derselben Warteschlangen-Idee, nur merken wir uns, von wem aus wir jemanden erreicht haben (`vorgaenger`).

In [ ]:
def finde_konflikte(zimmer_liste, getrennt):
    """Gibt alle Paare (a, b) zurück, die in einem Zimmer sind, aber nicht zusammen sein dürfen (jedes Paar einmal)."""
    konflikte = []
    for zimmer in zimmer_liste:
        for i in range(len(zimmer)):
            for j in range(i + 1, len(zimmer)):
                if zimmer[j] in getrennt[zimmer[i]]:
                    konflikte.append((zimmer[i], zimmer[j]))
    return konflikte


def finde_kette(a, b, zusammen):
    """Kürzeste Kette a - ... - b über 'muss zusammen'-Verbindungen (Breitensuche mit Vorgängern)."""
    vorgaenger = {a: None}
    warteschlange = [a]
    while len(warteschlange) > 0:
        n = warteschlange.pop(0)
        for m in sorted(zusammen[n]):
            if m not in vorgaenger:
                vorgaenger[m] = n
                warteschlange.append(m)
    kette = [b]
    while vorgaenger[kette[-1]] is not None:
        kette.append(vorgaenger[kette[-1]])
    kette.reverse()
    return kette


zimmer_liste1 = bilde_zimmer(namen1, zusammen1)
for a, b in finde_konflikte(zimmer_liste1, getrennt1):
    print(a, 'und', b, 'dürfen nicht zusammen sein, müssen es aber:', ' - '.join(finde_kette(a, b, zusammen1)))

Anna und Paula dürfen nicht zusammen sein, müssen es aber: Anna - Paula
Dani und Lotta dürfen nicht zusammen sein, müssen es aber: Dani - Steffi - Lotta


## Schritt 5: Alles zusammen

Die Funktion `loese` fasst die Schritte zusammen und gibt die Zimmer zurück, oder `None` (keine Belegung möglich) und die Konflikte. `zimmerbelegung` liest eine Datei, ruft `loese` auf und druckt das Ergebnis lesbar. Einzelzimmer fassen wir in einer Zeile zusammen, damit die Ausgabe kurz bleibt.

In [ ]:
def loese(namen, will, nicht):
    """Gibt (zimmer_liste, konflikte, zusammen) zurück. zimmer_liste ist None, wenn nicht alle Wünsche erfüllbar sind."""
    zusammen, getrennt = beidseitig(namen, will, nicht)
    zimmer_liste = bilde_zimmer(namen, zusammen)
    konflikte = finde_konflikte(zimmer_liste, getrennt)
    if len(konflikte) > 0:
        return None, konflikte, zusammen
    return zimmer_liste, [], zusammen


def zimmerbelegung(dateiname):
    namen, will, nicht, _ = lies_wuensche(dateiname)
    zimmer_liste, konflikte, zusammen = loese(namen, will, nicht)
    if zimmer_liste is None:
        print('Nicht alle Wünsche können erfüllt werden:')
        for a, b in konflikte:
            print('  ', a, 'und', b, 'dürfen nicht zusammen sein, müssen es aber:', ' - '.join(finde_kette(a, b, zusammen)))
        return None
    print('Alle Wünsche können erfüllt werden,', len(zimmer_liste), 'Zimmer:')
    einzel = []
    for zimmer in sorted(zimmer_liste, key=len, reverse=True):
        if len(zimmer) == 1:
            einzel.append(zimmer[0])
        else:
            print('  ', len(zimmer), 'Schülerinnen:', ', '.join(sorted(zimmer)))
    if len(einzel) > 0:
        print('  ', len(einzel), 'Einzelzimmer:', ', '.join(sorted(einzel)))
    return zimmer_liste


for nummer in range(1, 3):
    print('--- zimmerbelegung' + str(nummer))
    zimmerbelegung('beispieldaten/zimmerbelegung' + str(nummer) + '.txt')

--- zimmerbelegung1
Nicht alle Wünsche können erfüllt werden:
   Anna und Paula dürfen nicht zusammen sein, müssen es aber: Anna - Paula
   Dani und Lotta dürfen nicht zusammen sein, müssen es aber: Dani - Steffi - Lotta
--- zimmerbelegung2
Alle Wünsche können erfüllt werden, 3 Zimmer:
   3 Schülerinnen: Emma, Mia, Zoe
   2 Schülerinnen: Alina, Lilli
   1 Einzelzimmer: Lara


Beispiel 1 hat keine Lösung (die Gründe stehen da). Beispiel 2 hat eine: ein Dreier-, ein Zweier- und ein Einzelzimmer. Nun die vier großen Beispiele mit je 43 Schülerinnen:

In [ ]:
for nummer in range(3, 7):
    print('--- zimmerbelegung' + str(nummer))
    zimmerbelegung('beispieldaten/zimmerbelegung' + str(nummer) + '.txt')

--- zimmerbelegung3
Alle Wünsche können erfüllt werden, 5 Zimmer:
   14 Schülerinnen: Anna, Antonia, Carolin, Emily, Emma, Jana, Johanna, Julia, Larissa, Lisa, Marie, Michelle, Nele, Sofia
   13 Schülerinnen: Alina, Annika, Josephine, Katharina, Kim, Leonie, Lilli, Melina, Pauline, Pia, Sarah, Sophie, Vanessa
   10 Schülerinnen: Celina, Charlotte, Jasmin, Jessika, Lara, Laura, Luisa, Merle, Miriam, Nina
   5 Schülerinnen: Celine, Clara, Lea, Lena, Lina
   1 Einzelzimmer: Hannah
--- zimmerbelegung4
Alle Wünsche können erfüllt werden, 4 Zimmer:
   19 Schülerinnen: Annika, Carolin, Celina, Celine, Charlotte, Clara, Emma, Hannah, Kim, Lara, Lina, Lisa, Luisa, Merle, Michelle, Nele, Nina, Pia, Vanessa
   17 Schülerinnen: Alina, Emily, Jana, Johanna, Josephine, Katharina, Larissa, Laura, Lea, Lena, Leonie, Lilli, Marie, Melina, Sarah, Sofia, Sophie
   6 Schülerinnen: Anna, Jasmin, Jessika, Julia, Miriam, Pauline
   1 Einzelzimmer: Antonia
--- zimmerbelegung5
Alle Wünsche können erfüllt werde

Alle vier großen Beispiele sind lösbar. Beachte, wie unterschiedlich sie aussehen: Beispiel 5 hat fast nur Einzelzimmer (32 Schülerinnen werden durch keinen „Zusammen“-Wunsch gebunden), Beispiel 6 dagegen ein großes Zimmer mit 25 Schülerinnen.

**Wichtig:** Das sind die Zimmer, die **mindestens** nötig sind, um die „Zusammen“-Wünsche zu erfüllen, also die **meisten** Zimmer. Wer lieber weniger, größere Zimmer hätte, kann Zimmer noch zusammenlegen, siehe „Für Neugierige“.

## Kontrolle: Stimmt das wirklich?

Zwei unabhängige Tests:

1. **`erfuellt`** prüft eine Belegung direkt gegen die **ursprünglichen** Listen (ohne die beidseitige Umformung): Jede Schülerin muss mit allen aus `will` im Zimmer sein und mit keiner aus `nicht`.
2. **Alle Belegungen durchprobieren** (Brute Force) bei kleinen zufälligen Klassen mit 2 bis 7 Schülerinnen: Das Programm muss genau dann eine Belegung melden, wenn es irgendeine gibt. Für 7 Schülerinnen gibt es 877 Möglichkeiten, eine Menge in Gruppen aufzuteilen; bei 43 wären es unvorstellbar viele, deshalb gilt das nur als Test.

In [ ]:
import random


def erfuellt(zimmer_liste, will, nicht):
    """Prüft eine Belegung direkt gegen die ursprünglichen Wunschlisten."""
    nummer = {}
    for i in range(len(zimmer_liste)):
        for n in zimmer_liste[i]:
            nummer[n] = i
    for n in will:
        for m in will[n]:
            if nummer[n] != nummer[m]:
                return False
        for m in nicht[n]:
            if nummer[n] == nummer[m]:
                return False
    return True


def alle_aufteilungen(liste):
    """Erzeugt alle Möglichkeiten, die Liste in Gruppen aufzuteilen (jede als Liste von Listen)."""
    if len(liste) == 0:
        return [[]]
    ergebnis = []
    erstes = liste[0]
    for rest_aufteilung in alle_aufteilungen(liste[1:]):
        ergebnis.append([[erstes]] + rest_aufteilung)                  # erstes allein
        for i in range(len(rest_aufteilung)):                          # oder zu einer vorhandenen Gruppe
            neu = [gruppe[:] for gruppe in rest_aufteilung]
            neu[i].append(erstes)
            ergebnis.append(neu)
    return ergebnis


zufall = random.Random(2017)
loesbar = 0
unloesbar = 0
fehler = 0
for versuch in range(400):
    n = zufall.randint(2, 7)
    namen_t = ['S' + str(i) for i in range(n)]
    will_t = {x: set() for x in namen_t}
    nicht_t = {x: set() for x in namen_t}
    dichte = zufall.choice([0.05, 0.1, 0.2])
    for x in namen_t:
        for y in namen_t:
            if x != y:
                zahl = zufall.random()
                if zahl < dichte:
                    will_t[x].add(y)
                elif zahl < 2 * dichte:
                    nicht_t[x].add(y)
    gibt_es = any(erfuellt(a, will_t, nicht_t) for a in alle_aufteilungen(namen_t))
    gefunden, _, _ = loese(namen_t, will_t, nicht_t)
    if gefunden is not None and not erfuellt(gefunden, will_t, nicht_t):
        fehler += 1                                # behauptet eine Belegung, die nicht stimmt
    if (gefunden is not None) != gibt_es:
        fehler += 1                                # Existenz falsch beurteilt
    if gibt_es:
        loesbar += 1
    else:
        unloesbar += 1

for nummer in range(2, 7):
    namen_b, will_b, nicht_b, _ = lies_wuensche('beispieldaten/zimmerbelegung' + str(nummer) + '.txt')
    belegung_b, _, _ = loese(namen_b, will_b, nicht_b)
    print('zimmerbelegung' + str(nummer) + ': erfüllt alle ursprünglichen Wünsche:', erfuellt(belegung_b, will_b, nicht_b))
print('Zufallstests:', loesbar, 'lösbar,', unloesbar, 'unlösbar, Fehler:', fehler)

zimmerbelegung2: erfüllt alle ursprünglichen Wünsche: True
zimmerbelegung3: erfüllt alle ursprünglichen Wünsche: True
zimmerbelegung4: erfüllt alle ursprünglichen Wünsche: True
zimmerbelegung5: erfüllt alle ursprünglichen Wünsche: True
zimmerbelegung6: erfüllt alle ursprünglichen Wünsche: True
Zufallstests: 267 lösbar, 133 unlösbar, Fehler: 0


Das Verfahren hat in allen Zufallstests richtig entschieden, und die fünf lösbaren Beispiele erfüllen auch die ursprünglichen Listen.

## Für Neugierige: Möglichst wenige Zimmer

Die Aufgabe verlangt nur *irgendeine* Belegung. Unser Verfahren liefert die mit den **meisten** Zimmern (jede Schülerin sitzt nur dort, wo sie sitzen *muss*). In Beispiel 5 sind das 35 Zimmer, obwohl vielen Schülerinnen egal ist, wer neben ihnen wohnt. Die Lösungshinweise bemerken, dass die Belegung mit den *wenigsten* Zimmern **nicht eindeutig** ist: Bei Ada, Grace und Sheila, wo nur Ada und Sheila nicht zusammen wollen, kann man Ada und Grace zusammenlegen oder Grace und Sheila, aber nicht alle drei.

Der Ansatz: Wir legen zwei Pflichtzimmer zusammen, wenn niemand aus dem einen mit jemandem aus dem anderen ein „auf keinen Fall“ hat. Ein einfaches **gieriges Verfahren**: Der Reihe nach (große Zimmer zuerst) jedes Pflichtzimmer in das erste bisherige Zimmer stecken, mit dem es sich verträgt, sonst ein neues Zimmer beginnen.

In [ ]:
def vertraegt_sich(zimmer_a, zimmer_b, getrennt):
    """True, wenn niemand aus zimmer_a mit jemandem aus zimmer_b ein 'auf keinen Fall' hat."""
    for a in zimmer_a:
        for b in zimmer_b:
            if b in getrennt[a]:
                return False
    return True


def lege_zusammen_gierig(zimmer_liste, getrennt):
    """Verteilt die Pflichtzimmer (größte zuerst) auf möglichst wenige Zimmer, ohne dass sich zwei ausschließen."""
    neu = []
    for pflicht in sorted(zimmer_liste, key=len, reverse=True):
        for zimmer in neu:
            if vertraegt_sich(zimmer, pflicht, getrennt):
                zimmer.extend(pflicht)
                break
        else:
            neu.append(pflicht[:])                  # passt in kein Zimmer: neues Zimmer
    return neu


GIERIG = {}
for nummer in range(2, 7):
    namen_g, will_g, nicht_g, _ = lies_wuensche('beispieldaten/zimmerbelegung' + str(nummer) + '.txt')
    pflicht_g, _, _ = loese(namen_g, will_g, nicht_g)
    _zusammen_g, getrennt_g = beidseitig(namen_g, will_g, nicht_g)
    gierig = lege_zusammen_gierig(pflicht_g, getrennt_g)
    GIERIG[nummer] = len(gierig)
    gueltig = erfuellt(gierig, will_g, nicht_g)
    print('zimmerbelegung' + str(nummer) + ':', len(pflicht_g), 'Pflichtzimmer ->', len(gierig), 'Zimmer, gültig:', gueltig,
          ', Größen:', sorted((len(z) for z in gierig), reverse=True))

zimmerbelegung2: 3 Pflichtzimmer -> 3 Zimmer, gültig: True , Größen: [3, 2, 1]
zimmerbelegung3: 5 Pflichtzimmer -> 4 Zimmer, gültig: True , Größen: [14, 13, 11, 5]
zimmerbelegung4: 4 Pflichtzimmer -> 4 Zimmer, gültig: True , Größen: [19, 17, 6, 1]
zimmerbelegung5: 35 Pflichtzimmer -> 2 Zimmer, gültig: True , Größen: [31, 12]
zimmerbelegung6: 5 Pflichtzimmer -> 3 Zimmer, gültig: True , Größen: [25, 11, 7]


Das gierige Verfahren findet gültige Belegungen, bei Beispiel 3, 5 und 6 mit weniger Zimmern. Ob das **das Minimum** ist, wissen wir damit nicht. Das ist ein bekanntes schwieriges Problem (**Graphfärbung**: Pflichtzimmer sind Knoten, ein „auf keinen Fall“ zwischen zwei Pflichtzimmern ist eine Kante, und zwei verbundene Pflichtzimmer dürfen nicht dieselbe Farbe, also dasselbe Zimmer, bekommen). Hier hilft MiniZinc.

## Zum Schluss: Die wenigsten Zimmer mit MiniZinc

Das Hauptproblem (alle Wünsche erfüllen) löst unser Python-Programm mit Flutfüllen in einem Durchgang; MiniZinc wäre dafür nicht besser. Für die **Zusatzfrage** „wie wenige Zimmer reichen?“ ist ein Solver aber genau das Richtige, weil wir nur beschreiben müssen, was gelten soll:

- `k` ist die Anzahl der Pflichtzimmer (aus unserem Python-Programm), `konflikt[a, b]` ein 2D-Array: `true`, wenn jemand aus Pflichtzimmer `a` nicht mit jemandem aus `b` zusammen sein will.
- `zimmer[a]` ist die Nummer des endgültigen Zimmers, in das Pflichtzimmer `a` kommt (Werte `1..k`).
- Ein Paar mit Konflikt darf nicht dieselbe Nummer haben: `zimmer[a] != zimmer[b]` (mit `forall … where …`).
- `anzahl` ist die größte benutzte Nummer; das **Minimum** davon ist die gesuchte Zimmerzahl.

Die Modelle sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den [MiniZinc Playground](https://play.minizinc.org/)).

Das Modell (`zimmer_minimal.mzn`):

```minizinc
int: k;                                       % Anzahl der Pflichtzimmer
array[1..k, 1..k] of bool: konflikt;          % true: zwischen den beiden Pflichtzimmern gibt es ein "auf keinen Fall"
array[1..k] of var 1..k: zimmer;              % Nummer des endgültigen Zimmers jedes Pflichtzimmers
var 1..k: anzahl;                             % Zahl der benutzten Zimmer

constraint forall(a in 1..k, b in a+1..k where konflikt[a, b])(zimmer[a] != zimmer[b]);
constraint forall(a in 1..k)(zimmer[a] <= anzahl);   % Zimmer 1..anzahl werden benutzt
solve minimize anzahl;

output ["\(anzahl)"];
```

Die Daten (`.dzn`) erzeugt eine Python-Zelle `als_dzn` aus den Pflichtzimmern; sie stellt auch die Konfliktmatrix auf (`array2d`):

In [ ]:
def als_dzn(pflicht, getrennt):
    """Pflichtzimmer und Konfliktmatrix als MiniZinc-Daten (.dzn)."""
    k = len(pflicht)
    eintraege = []
    for a in range(k):
        for b in range(k):
            konflikt = a != b and not vertraegt_sich(pflicht[a], pflicht[b], getrennt)
            eintraege.append('true' if konflikt else 'false')
    return 'k = ' + str(k) + ';' + chr(10) + 'konflikt = array2d(1..' + str(k) + ', 1..' + str(k) + ', [' + ', '.join(eintraege) + ']);'


namen2, will2, nicht2, _ = lies_wuensche('beispieldaten/zimmerbelegung2.txt')
pflicht2, _, _ = loese(namen2, will2, nicht2)
_z2, getrennt2 = beidseitig(namen2, will2, nicht2)
print(als_dzn(pflicht2, getrennt2))

k = 3;
konflikt = array2d(1..3, 1..3, [false, true, true, true, false, true, true, true, false]);


Ergebnisse (Solver Gecode, bei allen Beispielen in unter einer Sekunde fertig):

zimmerbelegung2 (3 Pflichtzimmer):

**Ausgabe:**

```
3
```

zimmerbelegung3 (5 Pflichtzimmer):

**Ausgabe:**

```
4
```

zimmerbelegung4 (4 Pflichtzimmer):

**Ausgabe:**

```
4
```

zimmerbelegung5 (35 Pflichtzimmer):

**Ausgabe:**

```
2
```

zimmerbelegung6 (5 Pflichtzimmer):

**Ausgabe:**

```
3
```

Zum Vergleich mit dem gierigen Verfahren (gierig / Minimum): Beispiel 2: 3 / 3, Beispiel 3: 4 / 4, Beispiel 4: 4 / 4, Beispiel 5: 2 / 2, Beispiel 6: 3 / 3. Bei diesen Beispielen ist das gierige Verfahren also schon optimal; das ist aber kein Beweis, dass es immer so ist, und erst der Solver gibt Gewissheit. Bei Beispiel 2 (drei Pflichtzimmer, die sich paarweise ausschließen) und Beispiel 4 lässt sich nichts zusammenlegen, bei Beispiel 3 und 6 spart man ein bzw. zwei Zimmer, und bei Beispiel 5 reichen **zwei** Zimmer statt 35.

Grenzen: Das Modell bestimmt nur die **Anzahl**; die Zuordnung (`zimmer[a]`) könnte man sich mit einer anderen `output`-Zeile ausgeben lassen. Und Graphfärbung ist im Allgemeinen schwer: Bei sehr vielen Pflichtzimmern mit vielen Konflikten könnte der Solver lange brauchen. Bei den Beispielen dieser Aufgabe ist das kein Problem.